# 📘 智能体架构 14：可观测性 + 空运行测试工具

欢迎来到我们系列中一个专注于 AI 智能体部署和操作安全的关键笔记本。我们将实现一个**可观测性和空运行测试工具**，这是用于测试、调试和安全管理与真实世界系统交互的智能体的基本模式。

核心原则简单而强大：**在确切知道智能体将要做什么之前，永远不要在实时环境中运行它的操作。**该架构将"三思而后行"的过程正式化。智能体首先在 `dry_run` 模式下执行其计划，该模式不会改变真实世界，但生成详细的日志和清晰的行动计划。然后，该计划将提交给人类（或自动检查器）进行批准，然后才允许最终实时执行。

为了演示这一点，我们将构建一个**企业社交媒体智能体**。该智能体的任务是创建和发布帖子。我们将看到空运行测试工具如何使我们能够：
1.  **生成拟议帖子：** AI 将根据提示创造性地起草帖子。
2.  **执行空运行：** 智能体将在沙盒的 `dry_run=True` 模式下调用 `publish` 函数，生成关于*将*发生什么的日志。
3.  **人工审查：** 人类操作员将看到确切的帖子内容和空运行跟踪。他们必须输入 `approve` 才能继续。
4.  **执行实时操作：** 只有在批准后，`publish` 函数才会再次被调用，这次使用 `dry_run=False`，以执行真实操作。

该模式是负责任智能体部署的基石，为在生产环境中安全操作 AI 提供了所需的透明度和控制。

### 定义
**可观测性和空运行测试工具**是一种测试和部署架构，用于拦截智能体的操作。它首先在"空运行"或"沙盒"模式下执行它们，模拟操作而不引起真实世界的影响。然后，生成的计划和日志被提交审查，只有在明确批准后才在实时环境中执行操作。

### 高层工作流程

1.  **智能体提议操作：** 智能体确定要执行的计划或特定工具调用（例如，`api.post_update(...)`）。
2.  **空运行执行：** 测试工具使用 `dry_run=True` 标志调用智能体的计划。底层工具被设计为识别此标志，仅输出它们*将*做什么，以及日志和跟踪。
3.  **收集可观测性数据：** 测试工具捕获提议的操作、空运行日志以及来自模拟的任何其他相关跟踪数据。
4.  **人工/自动审查：** 该可观测性数据呈现给审查者。人类可以检查正确性、安全性和与目标的一致性。自动系统可以运行策略违规或已知不良模式的检查。
5.  **通过/不通过决策：** 审查者做出 `approve` 或 `reject` 决策。
6.  **实时执行（通过时）：** 如果获得批准，测试工具重新执行智能体的操作，这次使用 `dry_run=False`，使其产生真实世界的影响。

### 适用场景 / 应用
*   **调试和测试：** 在开发中，确切了解智能体如何解释任务以及它正在采取什么操作，而不会产生副作用。
*   **生产验证和安全：** 作为生产中的永久功能，适用于任何可以修改状态、花费资金、发送通信或执行任何其他不可逆操作的智能体。
*   **智能体的 CI/CD：** 将空运行测试工具集成到自动化测试管道中，以在部署新版本之前验证智能体行为。

### 优缺点
*   **优点：**
    *   **最大透明度和安全性：** 提供智能体操作的清晰、可审计的预览，防止昂贵或尴尬的错误。
    *   **出色的调试：** 轻松跟踪智能体的逻辑和工具调用，而无需撤消真实世界的更改。
*   **缺点：**
    *   **延迟部署/执行：** 强制审查步骤（尤其是人工）会引入延迟，使其不适合实时应用程序。
    *   **需要工具支持：** 智能体使用的工具和 API 必须设计为支持 `dry_run` 模式。

## 阶段 0：基础与环境设置

标准库和环境变量设置。

In [ ]:
# !pip install -q -U langchain-openai langchain langgraph rich python-dotenv

In [ ]:
import os
import datetime
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

# Pydantic for data modeling
from pydantic import BaseModel, Field

# LangChain components
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# LangGraph components
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel

# --- API Key and Tracing Setup ---
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Dry-Run Harness (OpenAI)"

required_vars = ["OPENAI_API_KEY", "LANGCHAIN_API_KEY"]
for var in required_vars:
    if var not in os.environ:
        print(f"Warning: Environment variable {var} not set.")

print("Environment variables loaded and tracing is set up.")

## 阶段 1：构建环境和工具

该架构的核心是支持 `dry_run` 模式的工具。我们将创建一个简单的 `SocialMediaAPI` 类。其 `publish_post` 方法将根据 `dry_run` 标志表现不同，提供我们所需的可观测性。

In [3]:
console = Console()

# Structured model for the agent's proposed post
class SocialMediaPost(BaseModel):
    content: str = Field(description="The full text content of the social media post.")
    hashtags: List[str] = Field(description="A list of relevant hashtags, without the '#'.")

# The key component: A tool with a dry_run flag
class SocialMediaAPI:
    """A mock social media API that supports a dry-run mode."""
    
    def publish_post(self, post: SocialMediaPost, dry_run: bool = True) -> Dict[str, Any]:
        """Publishes a post to the social media feed."""
        timestamp = datetime.datetime.now().isoformat()
        hashtags_str = ' '.join([f'#{h}' for h in post.hashtags])
        full_post_text = f"{post.content}\n\n{hashtags_str}"
        
        if dry_run:
            # In dry-run mode, we don't execute, we just return the plan and logs
            log_message = f"[DRY RUN] At {timestamp}, would publish the following post:\n--- PREVIEW ---\n{full_post_text}\n--- END PREVIEW ---"
            console.print(Panel(log_message, title="[yellow]Dry Run Log[/yellow]", border_style="yellow"))
            return {"status": "DRY_RUN_SUCCESS", "log": log_message, "proposed_post": full_post_text}
        else:
            # In live mode, we execute the action
            log_message = f"[LIVE] At {timestamp}, successfully published post!"
            console.print(Panel(log_message, title="[green]Live Execution Log[/green]", border_style="green"))
            # Here you would have the actual API call, e.g., twitter_client.create_tweet(...)
            return {"status": "LIVE_SUCCESS", "log": log_message, "post_id": f"post_{hash(full_post_text)}"}

social_media_tool = SocialMediaAPI()
print("Dry-run capable SocialMediaAPI tool defined successfully.")

Dry-run capable SocialMediaAPI tool defined successfully.


## 阶段 2：使用 LangGraph 构建空运行测试工具

现在我们将构建完整的工作流程。图将管理流程的状态，从提议操作到空运行和审查步骤，最后根据人工批准进行条件执行。

In [ ]:
model = os.environ.get("OPENAI_API_MODEL", "gpt-4o")
base_url = os.environ.get("OPENAI_API_BASE_URL", "https://api.openai.com/v1")
llm = ChatOpenAI(model=model, base_url=base_url, temperature=0.5)

# LangGraph State
class AgentState(TypedDict):
    user_request: str
    proposed_post: Optional[SocialMediaPost]
    dry_run_log: Optional[str]
    review_decision: Optional[str] # 'approve' or 'reject'
    final_status: str

# Graph Nodes
def propose_post_node(state: AgentState) -> Dict[str, Any]:
    """The creative agent that drafts the social media post."""
    console.print("--- 📝 Social Media Agent Drafting Post ---")
    prompt = ChatPromptTemplate.from_template(
        "You are a creative and engaging social media manager for a major AI company. Based on the user's request, draft a compelling social media post, including relevant hashtags.\n\nRequest: {request}"
    )
    post_generator_llm = llm.with_structured_output(SocialMediaPost)
    chain = prompt | post_generator_llm
    proposed_post = chain.invoke({"request": state['user_request']})
    return {"proposed_post": proposed_post}

def dry_run_review_node(state: AgentState) -> Dict[str, Any]:
    """Performs the dry run and prompts for human review."""
    console.print("--- 🧐 Performing Dry Run & Awaiting Human Review ---")
    dry_run_result = social_media_tool.publish_post(state['proposed_post'], dry_run=True)
    
    # Present the plan for review
    review_panel = Panel(
        f"[bold]Proposed Post:[/bold]\n{dry_run_result['proposed_post']}",
        title="[bold yellow]Human-in-the-Loop: Review Required[/bold yellow]",
        border_style="yellow"
    )
    console.print(review_panel)
    
    # Get human approval
    decision = ""
    while decision.lower() not in ["approve", "reject"]:
        decision = console.input("Type 'approve' to publish or 'reject' to cancel: ")
        
    return {"dry_run_log": dry_run_result['log'], "review_decision": decision.lower()}

def execute_live_post_node(state: AgentState) -> Dict[str, Any]:
    """Executes the live post after approval."""
    console.print("--- ✅ Post Approved, Executing Live ---")
    live_result = social_media_tool.publish_post(state['proposed_post'], dry_run=False)
    return {"final_status": f"Post successfully published! ID: {live_result.get('post_id')}"}

def post_rejected_node(state: AgentState) -> Dict[str, Any]:
    """Handles the case where the post is rejected."""
    console.print("--- ❌ Post Rejected by Human Reviewer ---")
    return {"final_status": "Action was rejected by the reviewer and not executed."}

# Conditional Edge
def route_after_review(state: AgentState) -> str:
    """Routes to execution or rejection based on the human review."""
    return "execute_live" if state["review_decision"] == "approve" else "reject"

# Build the graph
workflow = StateGraph(AgentState)
workflow.add_node("propose_post", propose_post_node)
workflow.add_node("dry_run_review", dry_run_review_node)
workflow.add_node("execute_live", execute_live_post_node)
workflow.add_node("reject", post_rejected_node)

workflow.set_entry_point("propose_post")
workflow.add_edge("propose_post", "dry_run_review")
workflow.add_conditional_edges("dry_run_review", route_after_review, {"execute_live": "execute_live", "reject": "reject"})
workflow.add_edge("execute_live", END)
workflow.add_edge("reject", END)

dry_run_agent = workflow.compile()
print("Dry-Run Harness agent graph compiled successfully.")

## 阶段 3：演示

让我们测试完整的系统。首先，使用一个我们将批准的安全、标准请求。其次，使用一个可能产生风险帖子的更模糊的请求，我们将拒绝它。

In [5]:
def run_agent_with_harness(request: str):
    initial_state = {"user_request": request}
    # Note: You will be prompted to type in the console below the cell.
    result = dry_run_agent.invoke(initial_state)
    console.print(f"\n[bold]Final Status:[/bold] {result['final_status']}")

# Test 1: A safe post that we will approve.
console.print("--- ✅ Test 1: Safe Post (Approve) ---")
run_agent_with_harness("Draft a positive launch announcement for our new AI model, 'Nebula'.")

# Test 2: A risky post that we will reject.
console.print("\n--- ❌ Test 2: Risky Post (Reject) ---")
run_agent_with_harness("Draft a post that emphasizes how much better our new 'Nebula' model is than the competition.")

--- ✅ Test 1: Safe Post (Approve) ---


--- 📝 Social Media Agent Drafting Post ---
--- 🧐 Performing Dry Run & Awaiting Human Review ---


                             Dry Run Log                              
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ [DRY RUN] At 2024-06-25T12:00:00.000000, would publish the        ┃
┃ following post:                                                  ┃
┃ --- PREVIEW ---                                                  ┃
┃ We're thrilled to announce the launch of our new flagship AI     ┃
┃ model, 'Nebula'! It's set to revolutionize natural language      ┃
┃ understanding and generation. A new era of AI is here!           ┃
┃                                                                  ┃
┃ #AI #Innovation #LaunchDay #Tech #Nebula                         ┃
┃ --- END PREVIEW ---                                              ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

                   Human-in-the-Loop: Review Required                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Proposed Post:                                                   ┃
┃ We're thrilled to announce the launch of our new flagship AI     ┃
┃ model, 'Nebula'! It's set to revolutionize natural language      ┃
┃ understanding and generation. A new era of AI is here!           ┃
┃                                                                  ┃
┃ #AI #Innovation #LaunchDay #Tech #Nebula                         ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛Type 'approve' to publish or 'reject' to cancel: 

--- ✅ Post Approved, Executing Live ---


...

--- 📝 Social Media Agent Drafting Post ---
--- 🧐 Performing Dry Run & Awaiting Human Review ---


                             Dry Run Log                              
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ [DRY RUN] At 2024-06-25T12:00:01.000000, would publish the        ┃
┃ following post:                                                  ┃
┃ --- PREVIEW ---                                                  ┃
┃ Our new 'Nebula' AI is so advanced, it's basically going to      ┃
┃ make all our competitors obsolete. They just can't keep up.      ┃
┃ Get ready for the future.                                        ┃
┃                                                                  ┃
┃ #GameChanger #AI #Disruption #NoCompetition #FutureIsNow         ┃
┃ --- END PREVIEW ---                                              ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

                   Human-in-the-Loop: Review Required                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Proposed Post:                                                   ┃
┃ Our new 'Nebula' AI is so advanced, it's basically going to      ┃
┃ make all our competitors obsolete. They just can't keep up.      ┃
┃ Get ready for the future.                                        ┃
┃                                                                  ┃
┃ #GameChanger #AI #Disruption #NoCompetition #FutureIsNow         ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛Type 'approve' to publish or 'reject' to cancel: 

--- ❌ Post Rejected by Human Reviewer ---



Final Status: Action was rejected by the reviewer and not executed.


### 结果分析

该演示是测试工具价值的完美展示：

1.  **安全帖子：** 第一个请求很直接。智能体生成了一篇既专业又热情的帖子。空运行准确预览了将要发布的内容。我们批准了它，`[LIVE]` 日志确认执行了真实操作。该过程按预期工作。

2.  **风险帖子：** 第二个请求更模糊，可能被激进地解释。智能体被要求强调优势，起草了一篇傲慢且不专业的帖子（`make all our competitors obsolete`）。虽然智能体完成了其创意提示，但这并不是真实公司想要发布的信息。

这就是测试工具证明其价值的地方。空运行在发布之前就暴露了这种风险内容。人类审查者很容易识别出不恰当的语气并输入 `reject`。图正确地路由到 `post_rejected_node`，最终状态确认**没有采取实时操作。** 一个简单的、结构化的工作流程避免了一场潜在的公关危机。

这清楚地分离了智能体的创意但不可预测的生成与确定性的、受控的执行，提供了一个至关重要的安全层。

## 结论

在本笔记本中，我们已经构建了一个完整的**可观测性和空运行测试工具**。该架构不仅仅是一个功能，而是部署与真实世界交互的智能体的基础理念。通过强制执行 `提议 → 审查 → 执行` 循环，我们获得了关键好处：

- **透明度：** 我们在智能体执行之前确切知道它打算做什么。
- **控制：** 我们有一个人工在环（或自动规则引擎），对任何操作拥有最终的否决权。
- **安全性：** 我们防止意外、昂贵或有害的操作，从充满希望的执行转向自信的部署。

虽然此模式引入了延迟，但它提供的安全性和可靠性对于大多数真实世界的应用程序来说是不可协商的。对于任何希望构建强大、可信赖和生产就绪的智能体系统的开发人员来说，它都是一个必不可少的工具。